In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from folium.plugins import FloatImage
import plotly.express as px
import plotly.graph_objects as go
from branca.colormap import LinearColormap, StepColormap
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
gdf_hex   = gpd.read_file("../data/processed/barcelona_hex_scored.geojson")
df_barris = pd.read_csv("../data/processed/barcelona_barri_scores.csv")

print(f"Hex grid:     {gdf_hex.shape}")
print(f"Barri scores: {df_barris.shape}")
print(f"Score range:  {gdf_hex['food_access_score'].min():.1f} – {gdf_hex['food_access_score'].max():.1f}")

Hex grid:     (997, 14)
Barri scores: (72, 7)
Score range:  20.0 – 80.0


In [3]:
m1 = folium.Map(location=[41.3874, 2.1686], zoom_start=13, tiles="CartoDB positron")

colormap = LinearColormap(
    colors=["#d73027", "#f46d43", "#fee08b", "#a6d96a", "#1a9850"],
    vmin=gdf_hex["food_access_score"].min(),
    vmax=gdf_hex["food_access_score"].max(),
    caption="Food Access Score (0 = worst, 100 = best)"
)
colormap.add_to(m1)

# Build a proper GeoJSON FeatureCollection with properties
features = []
for _, row in gdf_hex.iterrows():
    features.append({
        "type": "Feature",
        "geometry": row["geometry"].__geo_interface__,
        "properties": {
            "barri_name":            str(row.get("barri_name", "")),
            "food_access_score":     round(float(row["food_access_score"]), 1),
            "catchment_store_count": int(row["catchment_store_count"]),
            "is_food_desert":        bool(row["is_food_desert"]),
        }
    })

geojson_data = {"type": "FeatureCollection", "features": features}

folium.GeoJson(
    geojson_data,
    style_function=lambda feature: {
        "fillColor":  colormap(feature["properties"]["food_access_score"]),
        "color":      "white",
        "weight":      0.3,
        "fillOpacity": 0.75
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["barri_name", "food_access_score", "catchment_store_count", "is_food_desert"],
        aliases=["Neighbourhood", "Access Score", "Stores in catchment", "Food Desert?"],
    )
).add_to(m1)

m1.save("../outputs/maps/04_food_access_choropleth.html")
print("Map 1 saved ✓")
m1

Map 1 saved ✓


In [4]:
# Assign income and access quartiles per hex
gdf_hex["income_q"] = pd.qcut(
    gdf_hex["income_index"].fillna(gdf_hex["income_index"].median()),
    q=3, labels=["Low", "Mid", "High"]
)
gdf_hex["access_q"] = pd.qcut(
    gdf_hex["food_access_score"],
    q=3, labels=["Low", "Mid", "High"]
)

bivariate_colors = {
    ("Low",  "Low"):  "#e8d6c0",
    ("Low",  "Mid"):  "#c8a882",
    ("Low",  "High"): "#a67c52",
    ("Mid",  "Low"):  "#b0c4c8",
    ("Mid",  "Mid"):  "#8fa8aa",
    ("Mid",  "High"): "#6a8c8e",
    ("High", "Low"):  "#6a9fb5",
    ("High", "Mid"):  "#4a7f96",
    ("High", "High"): "#1a5f78",
}

gdf_hex["bivariate_color"] = gdf_hex.apply(
    lambda r: bivariate_colors.get((str(r["income_q"]), str(r["access_q"])), "#cccccc"), axis=1
)

# Build FeatureCollection with properties
features2 = []
for _, row in gdf_hex.iterrows():
    features2.append({
        "type": "Feature",
        "geometry": row["geometry"].__geo_interface__,
        "properties": {
            "barri_name":        str(row.get("barri_name", "")),
            "income_index":      round(float(row["income_index"]), 0) if pd.notna(row["income_index"]) else 0,
            "food_access_score": round(float(row["food_access_score"]), 1),
            "income_q":          str(row["income_q"]),
            "access_q":          str(row["access_q"]),
            "bivariate_color":   str(row["bivariate_color"]),
        }
    })

geojson_data2 = {"type": "FeatureCollection", "features": features2}

m2 = folium.Map(location=[41.3874, 2.1686], zoom_start=13, tiles="CartoDB positron")

folium.GeoJson(
    geojson_data2,
    style_function=lambda feature: {
        "fillColor":  feature["properties"]["bivariate_color"],
        "color":      "white",
        "weight":      0.3,
        "fillOpacity": 0.75
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["barri_name", "income_index", "food_access_score", "income_q", "access_q"],
        aliases=["Neighbourhood", "Median Income (€)", "Access Score", "Income Tier", "Access Tier"],
    )
).add_to(m2)

legend_html = """
<div style="position:fixed;bottom:40px;left:40px;background:white;
            padding:12px;border-radius:8px;font-size:11px;
            box-shadow:2px 2px 6px rgba(0,0,0,0.3);z-index:1000;">
  <b>Income vs Food Access</b><br><br>
  <table cellspacing="2">
    <tr><td></td><td style="text-align:center;font-size:10px">Low<br>Access</td>
                 <td style="text-align:center;font-size:10px">Mid<br>Access</td>
                 <td style="text-align:center;font-size:10px">High<br>Access</td></tr>
    <tr><td style="font-size:10px">High<br>Income</td>
        <td style="background:#6a9fb5;width:22px;height:22px"></td>
        <td style="background:#4a7f96;width:22px;height:22px"></td>
        <td style="background:#1a5f78;width:22px;height:22px"></td></tr>
    <tr><td style="font-size:10px">Mid<br>Income</td>
        <td style="background:#b0c4c8;width:22px;height:22px"></td>
        <td style="background:#8fa8aa;width:22px;height:22px"></td>
        <td style="background:#6a8c8e;width:22px;height:22px"></td></tr>
    <tr><td style="font-size:10px">Low<br>Income</td>
        <td style="background:#e8d6c0;width:22px;height:22px"></td>
        <td style="background:#c8a882;width:22px;height:22px"></td>
        <td style="background:#a67c52;width:22px;height:22px"></td></tr>
  </table>
</div>
"""
m2.get_root().html.add_child(folium.Element(legend_html))
m2.save("../outputs/maps/04_bivariate_income_access.html")
print("Map 2 saved ✓")
m2

Map 2 saved ✓


In [5]:
from folium.plugins import HeatMap

m3 = folium.Map(location=[41.3874, 2.1686], zoom_start=13, tiles="CartoDB dark_matter")

# Load clipped POIs
gdf_pois = gpd.read_file("../data/processed/barcelona_pois_clipped.geojson")

heat_data = [[row.geometry.y, row.geometry.x] for _, row in gdf_pois.iterrows()]
HeatMap(heat_data, radius=12, blur=8, max_zoom=16).add_to(m3)

m3.save("../outputs/maps/04_store_density_heatmap.html")
print("Map 3 saved ✓")
m3

Map 3 saved ✓


In [8]:
df_plot = df_barris.dropna(subset=["avg_food_access_score"]).sort_values("avg_food_access_score")

# Top 10 worst + top 10 best
df_extremes = pd.concat([df_plot.head(10), df_plot.tail(10)])
df_extremes["color"] = df_extremes["avg_food_access_score"].apply(
    lambda x: "#d73027" if x < 40 else "#1a9850"
)

fig = go.Figure(go.Bar(
    x=df_extremes["avg_food_access_score"].round(1),
    y=df_extremes["barri_name"],
    orientation="h",
    marker_color=df_extremes["color"],
    text=df_extremes["avg_food_access_score"].round(1),
    textposition="outside"
))

fig.update_layout(
    title="Food Access Score by Neighbourhood — Barcelona's Best & Worst",
    xaxis_title="Avg Food Access Score (0–100)",
    yaxis_title="",
    height=700,
    template="plotly_white",
    xaxis=dict(range=[0, 85]),
    margin=dict(l=250)
)

fig.add_shape(type="line", x0=0, x1=85, y0=9.5, y1=9.5,
              line=dict(color="grey", dash="dash", width=1))

fig.write_html("../outputs/maps/04_barri_rankings.html")
fig.write_image("../outputs/maps/04_barri_rankings.png", scale=2)
print("Chart saved ✓")
fig.show()

Chart saved ✓


In [11]:
df_scatter = df_barris.dropna(subset=["avg_food_access_score", "median_income"])

# Only label the most interesting outliers
label_these = [
    "la Vila de Gràcia",           # best access
    "Vallvidrera, el Tibidabo i les Planes",  # worst access
    "Pedralbes",                   # richest, low access
    "Ciutat Meridiana",            # poorest
    "el Raval",                    # low income, high access
    "Sant Genís dels Agudells",    # second worst
    "les Tres Torres",             # high income anomaly
]

df_scatter["label"] = df_scatter["barri_name"].apply(
    lambda x: x if x in label_these else ""
)

fig2 = px.scatter(
    df_scatter,
    x="median_income",
    y="avg_food_access_score",
    text="label",
    color="avg_food_access_score",
    color_continuous_scale="RdYlGn",
    size="total_stores",
    size_max=30,
    hover_name="barri_name",        # full name appears on hover instead
    hover_data={
        "median_income": True,
        "avg_food_access_score": True,
        "total_stores": True,
        "label": False
    },
    labels={
        "median_income":         "Median Household Income (€)",
        "avg_food_access_score": "Avg Food Access Score",
        "total_stores":          "Total Stores"
    },
    title="Income vs Food Access Score by Neighbourhood (Spearman r=0.095, p=0.003)"
)

fig2.update_traces(textposition="top center", textfont_size=9)
fig2.update_layout(
    template="plotly_white",
    height=700,       # taller canvas
    width=1100,       # wider canvas
    showlegend=False,
    margin=dict(t=100, r=150, l=60, b=60)  # more top + right breathing room
)

fig2.add_annotation(
    x=0.05, y=0.95, xref="paper", yref="paper",
    text="<b>Key insight:</b> Food deserts are geography-driven,<br>not income-driven in Barcelona",
    showarrow=False, bgcolor="white", bordercolor="grey",
    borderwidth=1, font=dict(size=11)
)

fig2.write_html("../outputs/maps/04_income_vs_access_scatter.html")
fig2.write_image("../outputs/maps/04_income_vs_access_scatter.png", scale=2)
print("Scatter saved ✓")
fig2.show()

Scatter saved ✓
